In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("/content/deliveries_updated_mens_ipl_upto_2024.csv.zip")

In [3]:
df.head()

,matchId,inning,over_ball,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,extras,isWide,isNoBall,Byes,LegByes,Penalty,dismissal_kind,player_dismissed,date
0,335982,1,0.1,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,0,1,NaN,NaN,NaN,1.0,NaN,NaN,NaN,2008-04-18
1,335982,1,0.2,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18
2,335982,1,0.3,0,3,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18
3,335982,1,0.4,0,4,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18
4,335982,1,0.5,0,5,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18


In [4]:
df.shape

(260920, 20)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   matchId           260920 non-null  int64  
 1   inning            260920 non-null  int64  
 2   over_ball         260920 non-null  float64
 3   over              260920 non-null  int64  
 4   ball              260920 non-null  int64  
 5   batting_team      260920 non-null  object 
 6   bowling_team      260920 non-null  object 
 7   batsman           260920 non-null  object 
 8   non_striker       260920 non-null  object 
 9   bowler            260920 non-null  object 
 10  batsman_runs      260920 non-null  int64  
 11  extras            260920 non-null  int64  
 12  isWide            8381 non-null    float64
 13  isNoBall          1093 non-null    float64
 14  Byes              673 non-null     float64
 15  LegByes           4001 non-null    float64
 16  Penalty           2 

In [6]:
df.isnull().sum()

,0
matchId,0
inning,0
over_ball,0
over,0
ball,0
batting_team,0
bowling_team,0
batsman,0
non_striker,0
bowler,0


In [7]:
df["dismissal_kind"] = df["dismissal_kind"].fillna("Not Out")
df["player_dismissed"] = df["player_dismissed"].fillna("None")
df["extras"] = df["extras"].fillna(0)

In [8]:
df.isnull().sum()


,0
matchId,0
inning,0
over_ball,0
over,0
ball,0
batting_team,0
bowling_team,0
batsman,0
non_striker,0
bowler,0


In [9]:
df["is_wicket"] = df["player_dismissed"].apply(
    lambda x: 0 if x == "None" else 1
)

In [10]:
df[["player_dismissed", "is_wicket"]].head()


,player_dismissed,is_wicket
0,None,0
1,None,0
2,None,0
3,None,0
4,None,0


In [11]:
df.columns = df.columns.str.lower().str.strip()

In [12]:
df.rename(columns={
    "matchid": "match_id",
    "batsman": "batter",
    "batsman_runs": "runs_off_bat"
}, inplace=True)


In [13]:
team_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
    "Royal Challengers Bangalore": "RCB",
    "Royal Challengers Bengaluru": "RCB"
}

df["batting_team"] = df["batting_team"].replace(team_mapping)
df["bowling_team"] = df["bowling_team"].replace(team_mapping)


In [14]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")


In [15]:
# Fill missing values before type conversion
df["iswide"] = df["iswide"].fillna(0)
df["isnoball"] = df["isnoball"].fillna(0)

# Convert to integer
df["iswide"] = df["iswide"].astype(int)
df["isnoball"] = df["isnoball"].astype(int)

# Convert remaining numeric columns safely
df["runs_off_bat"] = df["runs_off_bat"].fillna(0).astype(int)
df["extras"] = df["extras"].fillna(0).astype(int)
df["is_wicket"] = df["is_wicket"].fillna(0).astype(int)



In [16]:
df.dtypes

,0
match_id,int64
inning,int64
over_ball,float64
over,int64
ball,int64
batting_team,object
bowling_team,object
batter,object
non_striker,object
bowler,object


In [17]:
df.duplicated().sum()

np.int64(3)

In [18]:
df.drop_duplicates(inplace=True)


In [19]:
df.shape

(260917, 21)

In [20]:
drop_cols = ["byes", "legbyes", "penalty"]

for col in drop_cols:
    if col in df.columns:
        df.drop(columns=col, inplace=True)


In [21]:
df = df.sort_values(
    by=["date", "match_id", "inning", "over_ball"]
).reset_index(drop=True)


In [22]:
df.head()

,match_id,inning,over_ball,over,ball,batting_team,bowling_team,batter,non_striker,bowler,runs_off_bat,extras,iswide,isnoball,dismissal_kind,player_dismissed,date,is_wicket
0,335982,1,0.1,0,1,Kolkata Knight Riders,RCB,SC Ganguly,BB McCullum,P Kumar,0,1,0,0,Not Out,None,2008-04-18,0
1,335982,1,0.2,0,2,Kolkata Knight Riders,RCB,BB McCullum,SC Ganguly,P Kumar,0,0,0,0,Not Out,None,2008-04-18,0
2,335982,1,0.3,0,3,Kolkata Knight Riders,RCB,BB McCullum,SC Ganguly,P Kumar,0,1,1,0,Not Out,None,2008-04-18,0
3,335982,1,0.4,0,4,Kolkata Knight Riders,RCB,BB McCullum,SC Ganguly,P Kumar,0,0,0,0,Not Out,None,2008-04-18,0
4,335982,1,0.5,0,5,Kolkata Knight Riders,RCB,BB McCullum,SC Ganguly,P Kumar,0,0,0,0,Not Out,None,2008-04-18,0


In [23]:
df.shape

(260917, 18)

In [24]:
df.to_csv("cleaned_deliveries.csv", index=False)


In [25]:
print("Cleaned dataset saved successfully!")


Cleaned dataset saved successfully!
